# Glacier run plots

Discharge, storage and glacier-surface plots of a SUMMA glacier run against its upland-only twin, from the
glacier cells of SYMFLUENCE's 5a notebook. Runs are read from `simulations/<EXPERIMENT_ID>` and
`simulations/<EXPERIMENT_ID>_glac` of the domain, as SYMFLUENCE writes them; the SYMFLUENCE configuration supplies
`EXPERIMENT_ID`, `HYDROLOGICAL_MODEL` and `DOMAIN_DEFINITION_METHOD`.

In [ ]:
import sys
from pathlib import Path
import yaml

HERE = Path.cwd().parent                      # summa_glacier_workflow
sys.path.insert(0, str(HERE))
from domain_config import load_config

DOMAIN = "wolverine"                          # or "gulkana"
SYMFLUENCE_CONFIG = Path("~/Research/Symfluence/SYMFLUENCE/0_config_files/config_active.yaml").expanduser()
c = load_config(HERE / "domains" / f"{DOMAIN}.yaml")
with open(SYMFLUENCE_CONFIG) as f:
    config = yaml.safe_load(f)
config["DOMAIN_NAME"] = c["domain"]
project_dir = Path(c["project_dir"])

## 3. Visualize results with all versions on same plot

In [ ]:
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap, Normalize

plot_folder = project_dir / "plots" / "results"
plot_folder.mkdir(parents=True, exist_ok=True)
settings = project_dir / "settings" / config['HYDROLOGICAL_MODEL']

if config['DOMAIN_DEFINITION_METHOD'] != 'lumped':
    folder = project_dir / "simulations" / config['EXPERIMENT_ID'] / "mizuRoute"
    folder_glac = project_dir / "simulations" / f"{config['EXPERIMENT_ID']}_glac" / "mizuRoute"
else: # lumped domain definition has no routing
    folder = project_dir / "simulations" / f"{config['EXPERIMENT_ID']}" / config['HYDROLOGICAL_MODEL']
    folder_glac = project_dir / "simulations" / f"{config['EXPERIMENT_ID']}_glac" / config['HYDROLOGICAL_MODEL']

def _build_model_output(base_folder: Path, scheme: str, suffix: str):
    file_path = base_folder / scheme / f"{config['EXPERIMENT_ID']}_{suffix}.nc"
    return [(f"{config['HYDROLOGICAL_MODEL']}", str(file_path))]

# Build outputs for each scheme and timestep aggregation
model_outputs_sunenthalpy = _build_model_output(folder, "sun5_enthalpyForm", "timestep")
model_outputs_glac_sunenthalpy = _build_model_output(folder_glac, "sun5_enthalpyForm", "timestep")
model_outputs_beenthalpy = _build_model_output(folder, "be1_enthalpyForm", "timestep")
model_outputs_glac_beenthalpy = _build_model_output(folder_glac, "be1_enthalpyForm", "timestep")

model_outputs_sunenthalpy_day = _build_model_output(folder, "sun5_enthalpyForm", "day")
model_outputs_glac_sunenthalpy_day = _build_model_output(folder_glac, "sun5_enthalpyForm", "day")
model_outputs_beenthalpy_day = _build_model_output(folder, "be1_enthalpyForm", "day")
model_outputs_glac_beenthalpy_day = _build_model_output(folder_glac, "be1_enthalpyForm", "day")

# Annual outputs
model_outputs_sunenthalpy_annual = _build_model_output(folder, "sun5_enthalpyForm", "annual")
model_outputs_glac_sunenthalpy_annual = _build_model_output(folder_glac, "sun5_enthalpyForm", "annual")
model_outputs_beenthalpy_annual = _build_model_output(folder, "be1_enthalpyForm", "annual")
model_outputs_glac_beenthalpy_annual = _build_model_output(folder_glac, "be1_enthalpyForm", "annual")

obs_files = [('Observed', str(project_dir / "observations" / "streamflow" / "preprocessed" / f"{config.get('DOMAIN_NAME')}_streamflow_processed.csv"))]
storage_files = [('GRACE TWS', str(project_dir / "observations" / "storage" / "grace" / f"{config.get('DOMAIN_NAME')}_HRUs_GRUs_grace_tws_anomaly.csv"))]

In [ ]:
def get_NSE(obs,sim, transfo = 1): 

    obs = np.array(obs)
    sim = np.array(sim)

    isNotNA = np.invert(np.logical_or(np.isnan(obs), np.isnan(sim)))

    obs = obs[isNotNA]
    sim = sim[isNotNA]

    if transfo < 0:
        epsilon =  np.mean(obs)/100
    else:
        epsilon = 0

    obs    = (epsilon + obs) ** transfo
    sim    = (epsilon + sim) ** transfo

    nse    = 1-(np.sum(np.subtract(obs,sim)**2)/np.sum(np.subtract(obs, np.mean(sim))**2))

    return nse

In [ ]:
def correlation(obs,sim,method: str = 'pearson'):
    isNotNA = np.invert(np.logical_or(np.isnan(obs), np.isnan(sim)))

    obs = obs[isNotNA]
    sim = sim[isNotNA]

    if len(obs) < 2:
        return np.nan

    if method == 'pearson':
        if np.std(obs) == 0 or np.std(sim) == 0:
            return np.nan
        return float(np.corrcoef(obs, sim)[0, 1])
    elif method == 'spearman':
        return float(stats.spearmanr(obs, sim)[0])
    else:
        raise ValueError(f"Unknown correlation method: {method}")

In [ ]:
def get_KGE(obs,sim, transfo = 1): 
    obs = np.array(obs)
    sim = np.array(sim)

    isNotNA = np.invert(np.logical_or(np.isnan(obs), np.isnan(sim)))

    obs = obs[isNotNA]
    sim = sim[isNotNA]
    
    if transfo < 0:
        epsilon =  np.mean(obs)/100
    else:
        epsilon = 0

    obs    = (epsilon + obs) ** transfo
    sim    = (epsilon + sim) ** transfo

    sd_sim = np.std(sim, ddof=1)
    sd_obs = np.std(obs, ddof=1)
    
    m_sim  = np.mean(sim)
    m_obs  = np.mean(obs)
    
    r      = (np.corrcoef(sim,obs))[0,1]
    var    = float(sd_sim)/float(sd_obs)
    bias   = float(m_sim)/float(m_obs)
    
    kge    = 1.0-np.sqrt((r-1)**2 +(var-1)**2 + (bias-1)**2)

    return kge

In [ ]:
def get_RMSE(obs,sim, transfo = 1): 
    obs = np.array(obs)
    sim = np.array(sim)

    isNotNA = np.invert(np.logical_or(np.isnan(obs), np.isnan(sim)))

    obs = obs[isNotNA]
    sim = sim[isNotNA]

    if transfo < 0:
        epsilon =  np.mean(obs)/100
    else:
        epsilon = 0

    obs    = (epsilon + obs) ** transfo
    sim    = (epsilon + sim) ** transfo

    rmse   = np.sqrt(np.mean((obs-sim)**2))

    return rmse

In [ ]:
def get_CORR(obs,sim, method: str = 'pearson'):
    return correlation(obs, sim, method)    

In [ ]:
def calculate_metrics(obs: np.ndarray, sim: np.ndarray, is_storage_data: bool) -> dict:
    if is_storage_data:
        return {
            'RMSE': get_RMSE(obs, sim, transfo=1),
            'CORR': get_CORR(obs, sim, method='pearson'),
        }
    else:
        return {
            #'RMSE': get_RMSE(obs, sim, transfo=1),
            'KGE': get_KGE(obs, sim, transfo=1),
            #'KGEp': get_KGEp(obs, sim, transfo=1),
            'NSE': get_NSE(obs, sim, transfo=1),
            #'MAE': get_MAE(obs, sim, transfo=1),
            #'KGEnp': get_KGEnp(obs, sim, transfo=1)
        }

In [ ]:
def _add_overall_metrics(ax, obs_data, sim_data, is_storage_data):
     """Helper method to add overall metrics to the plot"""
     # Convert DataFrames to items format for storage data
     if is_storage_data:
         sim_items = list(sim_data.items())
         obs_items = list(obs_data.items())
     else:
         sim_items = sim_data
         obs_items = obs_data
     
     for i, (sim_name, sim) in enumerate(sim_items):
         if is_storage_data:
             sim_data0 = sim
             obs_name, obs_data0 = obs_items[0]  # Use first observation (usually only one)
         else:
             sim.index = sim.index.round(freq='h')
             if config['DOMAIN_DEFINITION_METHOD'] != 'lumped':
                 met_var = 'IRFroutedRunoff'
             else: # lumped domain definition has no routing, assume 1 GRU
                 met_var = 'averageRoutedRunoff'
             sim_data0 = sim[met_var]
             obs_data0 = obs_data[0][1]
         
         aligned_data = pd.merge(obs_data0, sim_data0, 
                                 left_index=True, right_index=True, how='inner')
         metrics = calculate_metrics(aligned_data.iloc[:, 0].values, aligned_data.iloc[:, 1].values, is_storage_data)
         
         metric_text = f"{sim_name} Metrics:n"
         metric_text += "n".join([f"{k}: {v:.3f} " for k, v in metrics.items()])
         
         ax.text(0.02, 0.98 - 0.15 * i, metric_text,
                 transform=ax.transAxes, verticalalignment='top', fontsize=8,
                 bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=3))

In [ ]:
anomaly_baseline = 'overlap'  # Example baseline setting
def _compute_anomaly(sim_series: pd.Series) -> pd.Series:
    """Helper method to compute anomaly of simulation series"""
    if anomaly_baseline == 'overlap':
        # Use mean of overlapping period
        baseline_mean = sim_series.mean()
    elif anomaly_baseline == 'full':
        # Use full simulation period (would need original full series)
        baseline_mean = sim_series.mean()
    elif '-' in str(anomaly_baseline):
        # Specific date range: 'YYYY-YYYY'
        try:
            start, end = anomaly_baseline.split('-')
            mask = (sim_series.index.year >= int(start)) & (sim_series.index.year <= int(end))
            if mask.any():
                baseline_mean = sim_series[mask].mean()
            else:
                baseline_mean = sim_series.mean()
        except:
            baseline_mean = sim_series.mean()
    else:
        baseline_mean = sim_series.mean()

    sim_anomaly = sim_series - baseline_mean
    return sim_anomaly

In [ ]:
# Read simulation data first
sim_data = []
name = ["be1Mixed","be1Mixed_glac"]#,"enthalpy", "enthalpy_glac"]
for i, model_output in enumerate([model_outputs_beenthalpy, model_outputs_glac_beenthalpy]):#,model_outputs_sunenthalpy, model_outputs_glac_sunenthalpy]): 
    for sim_name, sim_file in model_output:
        ds = xr.open_mfdataset(sim_file, engine='netcdf4')
        if config['DOMAIN_DEFINITION_METHOD'] != 'lumped':
            basinID = int(config.get('SIM_REACH_ID'))  
            segment_index = ds['reachID'].values == basinID
            ds = ds.sel(seg=ds['seg'][segment_index])
            str_var = 'IRFroutedRunoff'
        else: # lumped domain definition has no routing, assume 1 GRU
            basinID = 0 # assume 1 GRU
            ds = ds.sel(gru=ds['gru'][basinID])
            str_var = 'averageRoutedRunoff' # m/s
            area = ds['HRUarea'].values.sum() # m2
        df = ds[str_var].to_dataframe().reset_index()
        if config['DOMAIN_DEFINITION_METHOD'] == 'lumped':
            df['averageRoutedRunoff'] = df['averageRoutedRunoff'] * area
        df.set_index('time', inplace=True)
        sim_data.append((sim_name+name[i], df))

# Try to read observation data
obs_data = []
for obs_name, obs_file in obs_files:
    df = pd.read_csv(obs_file, parse_dates=['datetime'])
    df['datetime'] = pd.to_datetime(df['datetime'])
    df.set_index('datetime', inplace=True)
    df = df['discharge_cms'].resample('h').mean()
    obs_data.append((obs_name, df))

# Determine time range based on simulations, remove 180 days of spin-up
start_date = max([data.index.min() for _, data in sim_data]) + pd.Timedelta(180, unit="d") #+ pd.Timedelta(365, unit="d") 
end_date = min([data.index.max() for _, data in sim_data])

# Filter simulation data to closed time range
sim_data = [(name, data.loc[start_date:end_date]) for name, data in sim_data]

# Filter observation data if it exists
if obs_data:
    obs_data = [(name, data.loc[start_date:end_date]) for name, data in obs_data]

In [ ]:
# Read simulation data first
dostorage = True
if dostorage:
    if config['OPTIMIZATION_TARGET2'] == 'stor_mb': 
        files = [model_outputs_beenthalpy_day, model_outputs_glac_beenthalpy_day]
    else: 
        files = [model_outputs_beenthalpy, model_outputs_glac_beenthalpy]
    sim_data_stor = []
    for i, model_output in enumerate(files):
        for sim_name, sim_file in model_output:
            ds = xr.open_mfdataset(sim_file, engine='netcdf4')
            time_coord = None
            for var in ['time', 'Time', 'datetime']:
                if var in ds.coords or var in ds.dims:
                    time_coord = pd.to_datetime(ds[var].values)
                    break
            if config['DOMAIN_DEFINITION_METHOD'] != 'lumped':
                basinID = int(config.get('SIM_REACH_ID'))      
            else: # lumped domain definition has no routing, assume 1 GRU
                basinID = 0 # assume 1 GRU
            ds = ds.sel(gru=ds['gru'][basinID])

            if config['OPTIMIZATION_TARGET2'] == 'stor_mb':
                if 'basin__GlacierArea' in ds.variables and 'basin__GlacierStorage' in ds.variables:
                    area = ds['basin__GlacierArea']
                    area = np.where(area <= 0, np.nan, area)
                    total_tws = ds['basin__GlacierStorage']/area
                    # convert from Gt/m² (km³ of water/m² to mm
                    total_tws = total_tws * 1e9 * 1000.0
                else:
                    raise ValueError("Glacier mass balance variables (basin__GlacierArea, basin__GlacierStorage) not found in SUMMA output")
            else:
                if 'basin__StorageChange' in ds.data_vars:
                    total_tws = ds['basin__StorageChange']
                    total_tws = np.where(total_tws < -999, np.nan, total_tws)
                    # integrate: kg/m²/s * seconds -> mm per timestep, then cumulative sum over time
                    dt = config['FORCING_TIME_STEP_SIZE']
                    if hasattr(total_tws, 'sel') or hasattr(total_tws, 'dims'):
                        total_tws = (total_tws * dt).cumsum(dim=time_coord)
                    else:
                        total_tws = np.cumsum(total_tws * dt, axis=0)

            tws_series = pd.Series(total_tws.flatten(), index=time_coord, name='simulated_tws')
            ds.close()

            # change to monthly mean for storage comparison 
            df = tws_series.resample('MS').mean()
            sim_data_stor.append((sim_name+name[i], df))

    # read storage observation data, just do off GRACE for now
    obs_data_stor = []
    grace_column = 'grace_jpl_anomaly'  # Example column name, in cm since reference, monthly data
    for obs_name, obs_file in storage_files:
        df = pd.read_csv(obs_file, parse_dates=['Unnamed: 0'])
        df['datetime'] = pd.to_datetime(df['Unnamed: 0'])
        df.set_index('datetime', inplace=True)
        df = df[grace_column].resample('MS').mean()* 10 # convert cm to mm
        obs_data_stor.append((obs_name, df))

    # Remove 180 days of spin-up from simulation storage data
    sim_data_stor = [(name, data.loc[start_date:end_date]) for name, data in sim_data_stor]

    # Filter simulation and observation data to the shared time window
    sim_monthly = pd.concat({name: data.loc[start_date:end_date] for name, data in sim_data_stor}, axis=1)
    obs_monthly = pd.concat({name: data.loc[start_date:end_date] for name, data in obs_data_stor}, axis=1)

    # Find common period
    sim_monthly, obs_monthly = sim_monthly.align(obs_monthly, join='inner', axis=0)
    valid_mask = ~(sim_monthly.isna() | obs_monthly.isna())

    sim_matched = sim_monthly.where(valid_mask)
    obs_matched = obs_monthly.where(valid_mask)

    baseline_mean_sim = sim_matched.mean()
    baseline_mean_obs = obs_matched.mean()

    sim_anomaly = sim_matched - baseline_mean_sim
    obs_anomaly = obs_matched - baseline_mean_obs


In [ ]:
# anomaly
sim_monthly = pd.concat({name: series.resample('MS').mean() for name, series in sim_data_stor}, axis=1)
obs_monthly = pd.concat({name: series for name, series in obs_data_stor}, axis=1)
sim_monthly, obs_monthly = sim_monthly.align(obs_monthly, join='inner', axis=0)
valid_mask = ~(sim_monthly.isna() | obs_monthly.isna())
sim_matched = sim_monthly.where(valid_mask)
obs_matched = obs_monthly.where(valid_mask)
# Calculate anomalies
baseline_mean_sim = sim_matched.mean()
baseline_mean_obs = obs_matched.mean()
sim_anomaly = sim_matched - baseline_mean_sim
obs_anomaly = obs_matched - baseline_mean_obs

In [ ]:
# Create plot of streamflow comparison
fig, ax1 = plt.subplots(figsize=(10, 6))
fig.suptitle("Streamflow Comparison timeseries", fontsize=16, fontweight='bold')

# Plot time series
#colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
#linestyles = ['--', '-.', '-', ':', (0, (3, 5, 1, 5)), (0, (5, 10))]
colors = [ '#8c564b', '#2ca02c','#ff7f0e', '#1f77b4', '#d62728', '#9467bd'] 
linestyles = ['-', '-', '-', '-','-', '-']
              
# Plot observations if available
if obs_data:
    for obs_name, obs in obs_data:
        ax1.plot(obs.index, obs, label=f'{obs_name}', color='black', linewidth=2.5)

# Plot simulations
for (sim_name, sim), color, linestyle in zip(sim_data, colors, linestyles):
    ax1.plot(sim.index, sim[str_var], label=f'{sim_name}', 
            color=color, linestyle=linestyle, linewidth=1.5)
# Add metrics if observations are available
if obs_data:
    _add_overall_metrics(ax1, obs_data, sim_data, is_storage_data=False)

ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Streamflow (m³/s)', fontsize=12)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.set_facecolor('#f0f0f0')
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()
# Save the plot
fig.savefig(plot_folder / 'streamflow_comparison_all.png', dpi=300, bbox_inches='tight')


In [ ]:
# Create plot of storage comparison
if dostorage:
    fig, ax2 = plt.subplots(figsize=(10, 6))
    fig.suptitle("Storage Comparison timeseries", fontsize=16, fontweight='bold')

    # Plot observations if available
    if obs_anomaly.items():
        for obs_name, obs in obs_anomaly.items():
            ax2.plot(obs.index, obs, label=f'{obs_name}', color='black', linewidth=2.5)

    # Plot simulations
    for (sim_name, sim), color, linestyle in zip(sim_anomaly.items(), colors, linestyles):
        ax2.plot(sim.index, sim, label=f'{sim_name}', 
                color=color, linestyle=linestyle, linewidth=1.5)

    # Add metrics if observations are available
    if obs_anomaly.items():
        _add_overall_metrics(ax2, obs_anomaly, sim_anomaly, is_storage_data=True)

    ax2.set_xlabel('Date', fontsize=12)
    ax2.set_ylabel('Storage Anomaly (mm)', fontsize=12)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.set_facecolor('#f0f0f0')
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()
    # Save the plot
    fig.savefig(plot_folder / 'storage_comparison_all.png', dpi=300, bbox_inches='tight')

In [ ]:
# Create plot of streamflow comparison by day-of-year average
fig, ax1 = plt.subplots(figsize=(10, 6))
fig.suptitle("Streamflow Comparison day-of-year average", fontsize=16, fontweight='bold')

# Plot observations if available
if obs_data:
    for obs_name, obs in obs_data:
        obs = obs[obs.index.dayofyear != 366]
        day_of_year_avg = obs.groupby(obs.index.dayofyear).mean()
        ax1.plot(day_of_year_avg.index, day_of_year_avg, label=f'{obs_name}', color='black', linewidth=2.5)


# Plot simulations
for (sim_name, sim), color, linestyle in zip(sim_data, colors, linestyles):
    sim = sim[sim.index.dayofyear != 366]
    day_of_year_avg = sim.groupby(sim.index.dayofyear).mean()
    ax1.plot(day_of_year_avg.index, day_of_year_avg[str_var], label=f'{sim_name}', 
            color=color, linestyle=linestyle, linewidth=1.5)
    
# Add metrics if observations are available
if obs_data:
    _add_overall_metrics(ax1, obs_data, sim_data, is_storage_data=False)

ax1.set_xlabel('Day of Year', fontsize=12)
ax1.set_ylabel('Streamflow (m³/s)', fontsize=12)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, linestyle=':', alpha=0.55)
ax1.set_facecolor('#f0f0f0')
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

# Save the plot
fig.savefig(plot_folder / f'streamflow_comparison_all_wy.png', dpi=300, bbox_inches='tight')

In [ ]:
# Create plot of storage comparison by month-of-year average
if dostorage:
    fig, ax2 = plt.subplots(figsize=(10, 6))
    fig.suptitle("Storage Comparison month-of-year average", fontsize=16, fontweight='bold')

    # Plot observations if available
    if obs_anomaly.items():
        for obs_name, obs in obs_anomaly.items():
            obs = obs[obs.index.dayofyear != 366]
            month_of_year_avg = obs.groupby(obs.index.month).mean()
            # Append January again at position 13 for continuity
            month_of_year_avg_extended = pd.concat([month_of_year_avg, pd.Series({13: month_of_year_avg[1]})])
            ax2.plot(month_of_year_avg_extended.index, month_of_year_avg_extended.values, label=f'{obs_name}', color='black', linewidth=2.5)

    # Plot simulations
    for (sim_name, sim), color, linestyle in zip(sim_anomaly.items(), colors, linestyles):
        sim = sim[sim.index.dayofyear != 366]
        month_of_year_avg = sim.groupby(sim.index.month).mean()
        # Append January again at position 13 for continuity
        month_of_year_avg_extended = pd.concat([month_of_year_avg, pd.Series({13: month_of_year_avg[1]})])
        ax2.plot(month_of_year_avg_extended.index, month_of_year_avg_extended.values, label=f'{sim_name}', 
                color=color, linestyle=linestyle, linewidth=1.5)

    # Add metrics if observations are available
    if obs_anomaly.items():
        _add_overall_metrics(ax2, obs_anomaly, sim_anomaly, is_storage_data=True)

    ax2.set_xlabel('Month of Year', fontsize=12)
    ax2.set_ylabel('Storage Anomaly (mm)', fontsize=12)
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, linestyle=':', alpha=0.55)
    ax2.set_facecolor('#f0f0f0')
    ax2.set_xticks(range(1, 14))
    ax2.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

    # Save the plot
    fig.savefig(plot_folder / f'storage_comparison_all_wy.png', dpi=300, bbox_inches='tight')

In [ ]:
# Plot glacier surface change
# Parameters for plotting
viridis = plt.get_cmap('viridis')
viridis_r = viridis.reversed()
# Create a new colormap with 0 values set to white
colors = viridis_r(np.linspace(0, 1, 256))
colors[0] = [1, 1, 1, 1]  # Set the first color (0 value) to white
custom_cmap = LinearSegmentedColormap.from_list('custom_viridis_r', colors)
plot_diff_debris = False # plot difference in debris thickness instead of absolute values

In [ ]:
# Ensure axs1 and axs2 are 2D arrays even when nGlac==1 or nYears==1
def _ensure_2d(axs, rows, cols):
    # axs can be a single Axes, 1D array or 2D array
    if isinstance(axs, np.ndarray):
        if axs.ndim == 2:
            return axs
        if axs.ndim == 1:
            if rows == 1:
                return axs.reshape(1, -1)
            if cols == 1:
                return axs.reshape(-1, 1)
    else:
        # single Axes
        return np.array([[axs]])
    return axs

In [ ]:
# Loop over model outputs (maybe just one is fine)
for ii, model_output in enumerate([model_outputs_glac_beenthalpy_annual]):
    for sim_name, sim_file in model_output:
        ds = xr.open_mfdataset(sim_file, engine='netcdf4')
        time_coord = None
        for var in ['time', 'Time', 'datetime']:
            if var in ds.coords or var in ds.dims:
                time_coord = pd.to_datetime(ds[var].values)
                break
        if config['DOMAIN_DEFINITION_METHOD'] != 'lumped':
            basinID = int(config.get('SIM_REACH_ID'))      
        else: # lumped domain definition has no routing, assume 1 GRU
            basinID = 0 # assume 1 GRU
        ds = ds.sel(gru=ds['gru'][basinID])
        nYears = len(time_coord)
        # Get attributes for all glaciers in the basin
        if ii == 0:
            ds_attr = xr.open_mfdataset(settings / "attributes_glac.nc", engine='netcdf4')
            ds_attr = ds_attr.sel(gru=ds_attr['gru'][basinID])
        nGlac = int(ds_attr['nGrid'].values)

        # Create subplots for ice thickness and debris thickness
        fig1, axs1 = plt.subplots(nGlac, nYears, figsize=(3*nYears, 3*nGlac))
        fig1.suptitle("Ice Thickness over Time", fontsize=16, fontweight='bold')
        plt.tight_layout()
        if (nGlac>1): plt.subplots_adjust(top=0.93)
        axs1 = _ensure_2d(axs1, nGlac, nYears)
        fig2, axs2 = plt.subplots(nGlac, nYears, figsize=(3*nYears, 3*nGlac))
        if plot_diff_debris:
            fig2.suptitle("Debris Thickness Change (Yearly Difference)", fontsize=16, fontweight='bold')     
        else:
            fig2.suptitle("Debris Thickness over Time", fontsize=16, fontweight='bold')     
        plt.tight_layout()
        if (nGlac>1): plt.subplots_adjust(top=0.93)
        axs2 = _ensure_2d(axs2, nGlac, nYears)

        for i in range(nGlac):
            # get  bed topography and glacier mask
            glacID = int(ds['gridId'].values[i])
            dx = ds_attr['dx'].values[i]
            dy = ds_attr['dy'].values[i]
            nx = int(ds_attr['nx'].values[i])
            ny = int(ds_attr['ny'].values[i])
            x = np.arange(0, nx*dx, dx)
            y = np.arange(0, ny*dy, dy)
            bed_elev = ds_attr['bed_elev'].values[:, :, i]
            glacierMask = ds_attr['glacierMask'].values[:, :, i]
            rows, cols = np.where(glacierMask == 1)
            if len(rows) == 0 or len(cols) == 0:
                print(f"Skipping glacier {glacID} ({i+1} - no glacier mask")
                continue
            min_row, max_row = rows.min(), rows.max()
            min_col, max_col = cols.min(), cols.max()
            x = x[min_col:max_col+1]
            y = y[min_row:max_row+1]
            bed_elev = bed_elev[min_row:max_row+1, min_col:max_col+1]

            #get thickness and debris
            surface_elev = ds['surface_elev'].values[:, :, :, i]
            surface_elev = surface_elev[:,min_row:max_row+1, min_col:max_col+1]
            depths = surface_elev - bed_elev
            debris = ds['debris_thick'].values[:, :, :, i]
            debris = debris[:,min_row:max_row+1, min_col:max_col+1]
            times = pd.to_datetime(ds['time'].values)
            # subtract the year before from each year
            for jj in range(0, nYears):
                ax = axs1[i, jj]
                im0 = ax.imshow(depths[jj, :, :], extent=(x.min(), x.max(), y.min(), y.max()),
                                origin='lower', cmap=custom_cmap, norm=Normalize(vmin=0, vmax=np.nanmax(depths)))
                ax.invert_yaxis()
                if i == 0:
                    ax.set_title(times[jj].strftime('%Y-%m-%d'))
                if jj == 0:
                    ax.set_ylabel("Glacier ID " + str(glacID))
                else:
                    ax.set_yticklabels([])
                    ax.set_xticklabels([])
            # Add a single colorbar for the row
            fig1.colorbar(im0, ax=axs1[i, :], label='Thickness (m)', fraction=0.046, pad = 0.01,location='right')
            
            debris_diff = np.empty_like(debris)
            debris_diff[0,:,:] = debris[0,:,:] - debris[0,:,:]
            debris_diff[1:,:,:] = debris[1:,:,:] - debris[:-1,:,:]
            debris_diff[debris_diff == 0] = np.nan
            if plot_diff_debris: 
                debris = debris_diff
            vmax = np.nanmax(debris)
            vmin = np.nanmin(debris)
            for jj in range(0, nYears):
                min_thickness = 0.02
                if vmax == 0: vmax = min_thickness*2
                ax = axs2[i, jj]
                im0 = ax.imshow(debris[jj, :, :], extent=(x.min(), x.max(), y.min(), y.max()),
                                origin='lower', cmap=custom_cmap, norm=Normalize(vmin=vmin, vmax=vmax))
                ax.invert_yaxis()
                if i == 0:
                    ax.set_title(times[jj].strftime('%Y-%m-%d'))
                if jj == 0:
                    ax.set_ylabel("Glacier ID " + str(glacID))
                else:
                    ax.set_yticklabels([])
                    ax.set_xticklabels([])
            # Add a single colorbar for the row
            if plot_diff_debris:
                fig2.colorbar(im0, ax=axs2[i, :], label='Thickness Change (m)', fraction=0.046, pad = 0.01,location='right')     
            else:
                fig2.colorbar(im0, ax=axs2[i, :], label='Thickness (m)', fraction=0.046, pad = 0.01,location='right')

        # Save the plots
        fig1.savefig(plot_folder / f'iceThickness_GRU_{basinID}.png', dpi=300, bbox_inches='tight')
        fig2.savefig(plot_folder / f'debrisThickness_GRU_{basinID}.png', dpi=300, bbox_inches='tight')